# Semantic Kernel 

Trong ví dụ mã này, bạn sẽ sử dụng AI Framework [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) để tạo một agent đơn giản.

Mục đích của ví dụ này là minh họa các bước sẽ được áp dụng trong các đoạn mã sau này khi triển khai các mẫu agentic khác nhau.


## Nhập các Gói Python cần thiết


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## Tạo Client

Trong ví dụ này, chúng ta sẽ sử dụng [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) để truy cập LLM.

`ai_model_id` được đặt là `gpt-4o-mini`. Hãy thử chuyển mô hình sang một mô hình khác có sẵn trên thị trường GitHub Models để quan sát các kết quả khác nhau.

Để sử dụng `Azure Inference SDK` được yêu cầu cho `base_url` của GitHub Models, chúng ta sẽ tận dụng connector `OpenAIChatCompletion` bên trong Semantic Kernel. Ngoài ra cũng có [các connector khác có sẵn](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) cho phép bạn sử dụng Semantic Kernel với các nhà cung cấp mô hình khác.


In [ ]:
import random   

# Định nghĩa một plugin mẫu cho ví dụ

class DestinationsPlugin:
    """Danh sách các điểm đến ngẫu nhiên cho kỳ nghỉ."""

    def __init__(self):
        # Danh sách các điểm đến kỳ nghỉ
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Theo dõi điểm đến gần nhất để tránh lặp lại
        self.last_destination = None

    @kernel_function(description="Cung cấp một điểm đến kỳ nghỉ ngẫu nhiên.")
    def get_random_destination(self) -> Annotated[str, "Trả về một điểm đến kỳ nghỉ ngẫu nhiên."]:
        # Lấy các điểm đến có sẵn (loại trừ điểm đến gần nhất nếu có thể)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Chọn ngẫu nhiên một điểm đến
        destination = random.choice(available_destinations)

        # Cập nhật điểm đến gần nhất
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Tạo một AI Service sẽ được sử dụng bởi `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## Tạo Agent

Tại đây, chúng ta tạo Agent có tên là `TravelAgent`.

Trong ví dụ này, chúng ta sử dụng các chỉ thị rất cơ bản. Bạn có thể thoải mái sửa đổi các chỉ thị này để quan sát cách hành vi của agent thay đổi.


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="Bạn là một AI Agent hữu ích, có thể giúp lên kế hoạch kỳ nghỉ cho khách hàng tại các điểm đến ngẫu nhiên",
)

## Chạy các Agent

Bây giờ chúng ta có thể thực thi Agent bằng cách thiết lập `ChatHistory` và đưa `system_message` vào bên trong nó. Chúng ta sẽ sử dụng `AGENT_INSTRUCTIONS` mà chúng ta đã thiết lập trước đó.

Sau khi đã thiết lập xong, chúng ta tạo `user_inputs`, đại diện cho những gì người dùng đang gửi đến agent. Trong ví dụ này, thông báo được đặt thành `Plan me a day trip.` (Lên kế hoạch cho tôi một chuyến đi trong ngày).

Bạn có thể sửa đổi thông báo này để quan sát cách agent phản ứng khác đi.


In [ ]:
user_inputs = [
    "Lên kế hoạch cho tôi một chuyến đi trong ngày.",
    "Tôi không thích điểm đến đó. Lên kế hoạch cho tôi một kỳ nghỉ khác.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Bộ đệm (buffer) để tái tạo luồng gọi hàm
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Tích lũy các tham số (được stream thành từng phần)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Hoàn tất bất kỳ lệnh gọi hàm nào đang chờ trước khi hiển thị kết quả
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # giữ nguyên dưới dạng chuỗi thô

                        function_calls.append(f"Gọi hàm: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nKết quả hàm:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Các lệnh gọi hàm (nhấp để mở rộng)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()